In [1]:
import sys
from pathlib import Path

# Añade la raíz del repo al path para que `src.*` sea importable
# independientemente de desde dónde Jupyter arranque el kernel.
sys.path.insert(0, str(Path.cwd().parents[1]))

import matplotlib.pyplot as plt
import networkx as nx

from src.graphs.snap import SNAPDownloader


In [ ]:
downloader = SNAPDownloader(cache_dir="../../data/snap", verbose=True)
G = downloader.load_as_networkx("ego-Facebook")

# --- Métricas ---
is_directed = G.is_directed()
n = G.number_of_nodes()
m = G.number_of_edges()
degrees = [d for _, d in G.degree()]
avg_degree = sum(degrees) / n if n else 0
max_degree = max(degrees)
min_degree = min(degrees)
density = nx.density(G)

# Componentes y diámetro solo para grafos no dirigidos pequeños
if not is_directed:
    components = nx.number_connected_components(G)
    largest_cc = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(largest_cc)
    avg_clustering = nx.average_clustering(G)
    # Diámetro solo si la componente gigante es manejable
    diameter = nx.diameter(G_lcc) if len(largest_cc) <= 5000 else "N/A (grafo muy grande)"
else:
    components = nx.number_weakly_connected_components(G)
    avg_clustering = nx.average_clustering(G)
    diameter = "N/A (dirigido)"

print("=" * 45)
print(f"  Grafo: ego-Facebook")
print("=" * 45)
print(f"  Dirigido:            {is_directed}")
print(f"  Nodos:               {n:,}")
print(f"  Aristas:             {m:,}")
print(f"  Densidad:            {density:.6f}")
print(f"  Grado medio:         {avg_degree:.2f}")
print(f"  Grado máximo:        {max_degree}")
print(f"  Grado mínimo:        {min_degree}")
print(f"  Componentes:         {components}")
print(f"  Clustering medio:    {avg_clustering:.4f}")
print(f"  Diámetro (LCC):      {diameter}")
print("=" * 45)

pos = nx.spring_layout(G, seed=42, k=0.15)
fig, ax = plt.subplots(figsize=(12, 12), facecolor="#0f1419")
ax.set_facecolor("#0f1419")
nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_size=10, width=0.3, alpha=0.7,
    node_color="#4fc3f7", edge_color="#37474f",
    with_labels=False,
)
ax.set_title("ego-Facebook — red social", color="white", fontsize=14)
plt.tight_layout()
plt.show()